# Healthcare Readmission Risk Analysis

**Project:** Healthcare Readmission Risk Analysis  
**Author:** Daisy Omondi  
**Focus:** Python, SQL, healthcare operations, readmission patterns, decision support

## Purpose

This notebook shows how hospital admission data can be cleaned, structured and analyzed to identify 30-day readmission patterns.

The goal is not only to analyze data, but to connect the analysis to real healthcare workflow questions:

- Which patients show repeated admission patterns?
- Are previous admissions linked with higher readmission risk?
- How can structured data support earlier visibility for care teams?
- How can SQL and Python support operational decision-making?

## Data Privacy Note

This notebook uses simulated healthcare-style admission data for portfolio demonstration.  
No real patient data, no confidential hospital data and no personally identifiable patient information are included.


## 1. Import Libraries

In [ ]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

print("Libraries loaded successfully.")

## 2. Project Folder Setup

The notebook is written so it can run from the project root or from inside the `notebooks` folder.


In [ ]:
current_path = Path.cwd()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
VISUALS_DIR = PROJECT_ROOT / "visuals"
DATABASE_DIR = PROJECT_ROOT / "database"

for folder in [DATA_RAW_DIR, DATA_CLEANED_DIR, VISUALS_DIR, DATABASE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RAW_FILE = DATA_RAW_DIR / "hospital_admissions.csv"
CLEANED_FILE = DATA_CLEANED_DIR / "patient_admissions_clean.csv"
DATABASE_PATH = DATABASE_DIR / "healthcare_readmission.db"

print("Project root:", PROJECT_ROOT)
print("Raw data file:", RAW_FILE)
print("Cleaned data file:", CLEANED_FILE)
print("Database path:", DATABASE_PATH)

## 3. Create Simulated Healthcare Admission Data

This project does not use real patient data.  
The simulated dataset includes fields that are useful for demonstrating readmission analysis:

- `patient_id`
- `admission_id`
- `admission_date`
- `discharge_date`
- `age_group`
- `department`
- `previous_admissions`
- `length_of_stay`
- `readmitted_within_30_days`


In [ ]:
def create_simulated_admission_data(output_path: Path, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    previous_admission_groups = [0, 1, 2, 3, 4, 5]
    group_sizes = [1200, 1000, 850, 700, 500, 350]
    readmission_probabilities = {
        0: 0.078,
        1: 0.146,
        2: 0.283,
        3: 0.437,
        4: 0.612,
        5: 0.725,
    }

    departments = [
        "Internal Medicine",
        "Cardiology",
        "Neurology",
        "Surgery",
        "Emergency Observation",
        "Geriatrics",
    ]

    age_groups = ["18-39", "40-59", "60-74", "75+"]

    rows = []
    admission_counter = 1
    patient_counter = 10000
    start_date = pd.Timestamp("2024-01-01")

    for previous_admissions, group_size in zip(previous_admission_groups, group_sizes):
        for _ in range(group_size):
            patient_counter += 1
            patient_id = f"P{patient_counter}"
            admission_id = f"A{admission_counter:06d}"

            admission_date = start_date + pd.to_timedelta(int(rng.integers(0, 365)), unit="D")
            length_of_stay = int(rng.integers(1, 15))
            discharge_date = admission_date + pd.to_timedelta(length_of_stay, unit="D")

            readmitted = int(rng.random() < readmission_probabilities[previous_admissions])

            if readmitted == 1:
                days_to_readmission = int(rng.integers(2, 31))
            else:
                days_to_readmission = np.nan

            rows.append({
                "patient_id": patient_id,
                "admission_id": admission_id,
                "admission_date": admission_date,
                "discharge_date": discharge_date,
                "age_group": rng.choice(age_groups, p=[0.18, 0.28, 0.30, 0.24]),
                "department": rng.choice(departments),
                "previous_admissions": previous_admissions,
                "length_of_stay": length_of_stay,
                "days_to_readmission": days_to_readmission,
                "readmitted_within_30_days": readmitted,
            })

            admission_counter += 1

    df = pd.DataFrame(rows)

    # Add a few realistic data quality issues for the cleaning step.
    duplicate_rows = df.sample(20, random_state=seed)
    df = pd.concat([df, duplicate_rows], ignore_index=True)

    missing_department_index = df.sample(30, random_state=seed + 1).index
    df.loc[missing_department_index, "department"] = None

    df.to_csv(output_path, index=False)
    return df


if not RAW_FILE.exists():
    raw_df = create_simulated_admission_data(RAW_FILE)
    print("Simulated admission data created.")
else:
    raw_df = pd.read_csv(RAW_FILE)
    print("Existing raw data loaded.")

print(raw_df.shape)
raw_df.head()

## 4. Load Raw Data and Inspect Structure

In [ ]:
df_raw = pd.read_csv(RAW_FILE)

print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

df_raw.head()

In [ ]:
df_raw.info()

## 5. Data Quality Check Before Cleaning

In [ ]:
quality_before = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "duplicate_rows",
        "missing_values",
        "missing_departments",
        "missing_days_to_readmission",
    ],
    "value": [
        df_raw.shape[0],
        df_raw.shape[1],
        df_raw.duplicated().sum(),
        int(df_raw.isna().sum().sum()),
        int(df_raw["department"].isna().sum()),
        int(df_raw["days_to_readmission"].isna().sum()),
    ],
})

quality_before

## 6. Clean and Prepare Admission Data

Cleaning steps:

- Remove duplicate admission records
- Standardize admission and discharge dates
- Fill missing department values
- Ensure numeric fields are in the correct format
- Recalculate length of stay as a quality check
- Keep the 30-day readmission flag as the target indicator


In [ ]:
df = df_raw.copy()

df = df.drop_duplicates(subset=["admission_id"])

df["admission_date"] = pd.to_datetime(df["admission_date"], errors="coerce")
df["discharge_date"] = pd.to_datetime(df["discharge_date"], errors="coerce")

df["department"] = df["department"].fillna("Unknown")

numeric_columns = [
    "previous_admissions",
    "length_of_stay",
    "days_to_readmission",
    "readmitted_within_30_days",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.dropna(subset=["patient_id", "admission_id", "admission_date", "discharge_date"])

df["calculated_length_of_stay"] = (df["discharge_date"] - df["admission_date"]).dt.days
df["length_of_stay_check"] = df["length_of_stay"] == df["calculated_length_of_stay"]

df["readmitted_within_30_days"] = df["readmitted_within_30_days"].fillna(0).astype(int)
df["previous_admissions"] = df["previous_admissions"].fillna(0).astype(int)
df["length_of_stay"] = df["length_of_stay"].fillna(df["calculated_length_of_stay"]).astype(int)

df = df.sort_values(["patient_id", "admission_date"]).reset_index(drop=True)

print("Cleaned rows:", df.shape[0])
df.head()

## 7. Data Quality Check After Cleaning

In [ ]:
quality_after = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "duplicate_admission_ids",
        "missing_values_excluding_days_to_readmission",
        "invalid_length_of_stay_records",
        "readmission_flag_values",
    ],
    "value": [
        df.shape[0],
        df.shape[1],
        df["admission_id"].duplicated().sum(),
        int(df.drop(columns=["days_to_readmission"]).isna().sum().sum()),
        int((df["length_of_stay"] <= 0).sum()),
        sorted(df["readmitted_within_30_days"].unique().tolist()),
    ],
})

quality_after

## 8. Executive Summary KPIs

In [ ]:
total_admissions = len(df)
unique_patients = df["patient_id"].nunique()
readmissions_30d = df["readmitted_within_30_days"].sum()
readmission_rate = readmissions_30d / total_admissions * 100
average_length_of_stay = df["length_of_stay"].mean()
average_previous_admissions = df["previous_admissions"].mean()

kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Admissions",
        "Unique Patients",
        "30-Day Readmissions",
        "Readmission Rate (%)",
        "Average Length of Stay",
        "Average Previous Admissions",
    ],
    "Value": [
        total_admissions,
        unique_patients,
        readmissions_30d,
        round(readmission_rate, 2),
        round(average_length_of_stay, 2),
        round(average_previous_admissions, 2),
    ],
})

kpi_summary

## 9. Readmission Rate by Previous Admissions

This is the main analysis question in the project.

The aim is to understand whether patients with more previous admissions show higher 30-day readmission risk.


In [ ]:
readmission_summary = (
    df.groupby("previous_admissions")
    .agg(
        total_admissions=("admission_id", "count"),
        total_patients=("patient_id", "nunique"),
        readmissions_30d=("readmitted_within_30_days", "sum"),
        average_length_of_stay=("length_of_stay", "mean"),
    )
    .reset_index()
)

readmission_summary["readmission_rate"] = (
    readmission_summary["readmissions_30d"] / readmission_summary["total_admissions"] * 100
).round(1)

readmission_summary

## 10. Visualize the Readmission Pattern

In [ ]:
plt.figure(figsize=(9, 5))

x_labels = readmission_summary["previous_admissions"].astype(str).replace({"5": "5+"})
bars = plt.bar(x_labels, readmission_summary["readmission_rate"])

plt.title("Readmission Rate by Number of Previous Admissions")
plt.xlabel("Previous Patient Admissions")
plt.ylabel("Readmission Rate (%)")
plt.ylim(0, 100)

for bar, value in zip(bars, readmission_summary["readmission_rate"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 2,
        f"{value}%",
        ha="center",
    )

plt.tight_layout()

chart_path = VISUALS_DIR / "readmission_rate_chart_from_notebook.png"
plt.savefig(chart_path, dpi=200, bbox_inches="tight")
plt.show()

print("Chart saved to:", chart_path)

## 11. Readmission Pattern by Department

In [ ]:
department_summary = (
    df.groupby("department")
    .agg(
        total_admissions=("admission_id", "count"),
        readmissions_30d=("readmitted_within_30_days", "sum"),
        average_length_of_stay=("length_of_stay", "mean"),
    )
    .reset_index()
)

department_summary["readmission_rate"] = (
    department_summary["readmissions_30d"] / department_summary["total_admissions"] * 100
).round(1)

department_summary.sort_values("readmission_rate", ascending=False)

## 12. High-Risk Patient View

This table creates a simple operational view of patients who may need stronger follow-up after discharge.

In a real hospital environment, this would need clinical validation and privacy-safe governance before any operational use.


In [ ]:
high_risk_patients = (
    df[df["readmitted_within_30_days"] == 1]
    .sort_values(
        ["previous_admissions", "days_to_readmission", "length_of_stay"],
        ascending=[False, True, False],
    )
    [
        [
            "patient_id",
            "admission_id",
            "admission_date",
            "discharge_date",
            "department",
            "age_group",
            "previous_admissions",
            "length_of_stay",
            "days_to_readmission",
            "readmitted_within_30_days",
        ]
    ]
    .head(20)
)

high_risk_patients

## 13. SQL Analysis

This section loads the cleaned admissions table into a local SQLite database and runs SQL queries.

This shows that the analysis can be done not only in Python, but also in a query-ready SQL environment.


In [ ]:
df.to_csv(CLEANED_FILE, index=False)

with sqlite3.connect(DATABASE_PATH) as connection:
    df.to_sql("patient_admissions_clean", connection, if_exists="replace", index=False)

print("Cleaned data saved to CSV and SQLite database.")

In [ ]:
query_top_risk = """
SELECT
    patient_id,
    COUNT(*) AS total_admissions,
    SUM(readmitted_within_30_days) AS readmissions_30d,
    ROUND(100.0 * SUM(readmitted_within_30_days) / COUNT(*), 2) AS readmission_rate
FROM patient_admissions_clean
GROUP BY patient_id
HAVING COUNT(*) >= 1
ORDER BY readmission_rate DESC, total_admissions DESC
LIMIT 10;
"""

with sqlite3.connect(DATABASE_PATH) as connection:
    top_risk_patients_sql = pd.read_sql_query(query_top_risk, connection)

top_risk_patients_sql

In [ ]:
query_previous_admissions = """
SELECT
    previous_admissions,
    COUNT(*) AS total_admissions,
    SUM(readmitted_within_30_days) AS readmissions_30d,
    ROUND(100.0 * SUM(readmitted_within_30_days) / COUNT(*), 2) AS readmission_rate
FROM patient_admissions_clean
GROUP BY previous_admissions
ORDER BY previous_admissions;
"""

with sqlite3.connect(DATABASE_PATH) as connection:
    previous_admissions_sql = pd.read_sql_query(query_previous_admissions, connection)

previous_admissions_sql

In [ ]:
query_department = """
SELECT
    department,
    COUNT(*) AS total_admissions,
    SUM(readmitted_within_30_days) AS readmissions_30d,
    ROUND(100.0 * SUM(readmitted_within_30_days) / COUNT(*), 2) AS readmission_rate,
    ROUND(AVG(length_of_stay), 2) AS average_length_of_stay
FROM patient_admissions_clean
GROUP BY department
ORDER BY readmission_rate DESC;
"""

with sqlite3.connect(DATABASE_PATH) as connection:
    department_sql = pd.read_sql_query(query_department, connection)

department_sql

## 14. Operational Interpretation

The analysis suggests a clear pattern:

- Patients with more previous admissions show higher 30-day readmission risk.
- Previous utilization can be used as an early visibility signal.
- Short discharge-to-readmission gaps may point to follow-up or transition weaknesses.
- Structured data can help teams identify which patients may need closer review after discharge.

This project is not meant to replace clinical judgment.  
Its value is in supporting earlier visibility, better coordination and more structured operational decision-making.


## 15. Final Project Takeaway

This project connects technical analytics with healthcare operations.

From my healthcare background, I understand that readmissions are not just numbers in a table. They affect patients, nurses, hospital workflows, resource planning and quality outcomes.

The notebook shows how Python and SQL can help turn hospital admission data into practical decision-support insight.
